# Tiền xử lý dữ liệu PlantVillage

1. Đọc danh sách ảnh.
2. Kiểm tra và loại ảnh trùng.
3. Chia dữ liệu thành train, validation và test.
4. Chuyển ảnh sang RGB và resize trong bộ nhớ.
5. Lưu danh sách đã xử lý để các model sử dụng.

## 1. Import thư viện

In [1]:
from pathlib import Path
import hashlib
import os

import pandas as pd
from dotenv import load_dotenv
from PIL import Image
from sklearn.model_selection import train_test_split

SEED = 42
IMAGE_SIZE = 224

## 2. Khai báo đường dẫn

In [2]:
# Có thể chạy notebook từ thư mục dự án hoặc thư mục notebooks.
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

load_dotenv(PROJECT_DIR / ".env")
DATA_DIR = Path(
    os.getenv("DATA_DIR", PROJECT_DIR.parent / "PlantVillage")
).expanduser()
if not DATA_DIR.is_absolute():
    DATA_DIR = PROJECT_DIR / DATA_DIR
DATA_DIR = DATA_DIR.resolve()
MANIFEST_DIR = PROJECT_DIR / "data"
PREPROCESSING_PATH = MANIFEST_DIR / "preprocessing_data.csv"

print("Thư mục dữ liệu:", DATA_DIR)
print("File kết quả:", PREPROCESSING_PATH)

Thư mục dữ liệu: C:\Users\MY PC\Documents\DL\project 1\archive\PlantVillage
File kết quả: C:\Users\MY PC\Documents\DL\project 1\plant-disease-classification\data\preprocessing_data.csv


## 3. Đọc danh sách ảnh

In [3]:
rows = []
image_extensions = [".jpg", ".jpeg", ".png"]

for folder_name in ["train", "val"]:
    folder_path = DATA_DIR / folder_name

    for class_folder in sorted(folder_path.iterdir()):
        if not class_folder.is_dir():
            continue

        for image_path in class_folder.iterdir():
            if image_path.suffix.lower() in image_extensions:
                rows.append({
                    "relative_path": image_path.relative_to(DATA_DIR).as_posix(),
                    "class_name": class_folder.name,
                    "original_folder": folder_name,
                })

data = pd.DataFrame(rows)

print(f"Tổng số ảnh: {len(data):,}")
print(f"Số lớp: {data['class_name'].nunique()}")
data.head()

Tổng số ảnh: 54,305
Số lớp: 38


,relative_path,class_name,original_folder
0,train/Apple___Apple_scab/01a66316-0e98-4d3b-a5...,Apple___Apple_scab,train
1,train/Apple___Apple_scab/01f3deaa-6143-4b6c-9c...,Apple___Apple_scab,train
2,train/Apple___Apple_scab/0208f4eb-45a4-4399-90...,Apple___Apple_scab,train
3,train/Apple___Apple_scab/023123cb-7b69-4c9f-a5...,Apple___Apple_scab,train
4,train/Apple___Apple_scab/0261a6e4-21f8-481a-88...,Apple___Apple_scab,train


## 4. Kiểm tra và loại ảnh trùng

Hai ảnh có cùng mã MD5 là hai file giống hệt nhau. Ảnh đầu tiên được giữ lại, các bản sao được loại khỏi biến data.

In [ ]:
file_hashes = []

for index, relative_path in enumerate(data["relative_path"], start=1):
    image_path = DATA_DIR / relative_path
    file_hash = hashlib.md5(image_path.read_bytes()).hexdigest()
    file_hashes.append(file_hash)

    if index % 5000 == 0:
        print(f"Đã kiểm tra {index:,}/{len(data):,} ảnh")

data["file_hash"] = file_hashes

duplicate_images = data[data.duplicated(
    subset="file_hash",
    keep="first",
)].copy()

print(f"Số ảnh trùng: {len(duplicate_images):,}")

data = data.drop_duplicates(
    subset="file_hash",
    keep="first",
).reset_index(drop=True)

print(f"Số ảnh còn lại: {len(data):,}")
duplicate_images.head()

Đã kiểm tra 5,000/54,305 ảnh


Đã kiểm tra 10,000/54,305 ảnh


Đã kiểm tra 15,000/54,305 ảnh


Đã kiểm tra 20,000/54,305 ảnh


Đã kiểm tra 25,000/54,305 ảnh


...

Đã kiểm tra 40,000/54,305 ảnh


Đã kiểm tra 45,000/54,305 ảnh


Đã kiểm tra 50,000/54,305 ảnh


Số ảnh trùng: 21
Số ảnh còn lại: 54,284


,relative_path,class_name,original_folder,file_hash
1524,train/Apple___healthy/3673d121-b5de-481c-b057-...,Apple___healthy,train,68f05a541bf01c0fcfbc7f9445f8a147
2230,train/Apple___healthy/c21cf428-bfc3-4710-b5d2-...,Apple___healthy,train,161940bd6e73cf648974cc7213cffb8f
31840,train/Tomato___healthy/505465db-407b-4e0a-8110...,Tomato___healthy,train,2e53ad1ec3f810e2ff4c83400dc69ec9
32468,train/Tomato___healthy/cfd491d6-4af5-4728-8f0e...,Tomato___healthy,train,ea0a44df2e4e44c6ccacc31f067a6284
32478,train/Tomato___healthy/d2ce7896-6fa4-45e6-96c5...,Tomato___healthy,train,4b73700b68ce8704f6b4549a2846dede


## 5. Chia train, validation và test

- Ảnh trong thư mục train gốc được chia thành train và validation.
- Ảnh trong thư mục val gốc được dùng làm test.

In [5]:
train_data = data[data["original_folder"] == "train"].copy()
test_data = data[data["original_folder"] == "val"].copy()

train_data, validation_data = train_test_split(
    train_data,
    test_size=0.1,
    random_state=SEED,
    stratify=train_data["class_name"],
)

train_data["split"] = "train"
validation_data["split"] = "validation"
test_data["split"] = "test"

data = pd.concat(
    [train_data, validation_data, test_data],
    ignore_index=True,
)

print(data["split"].value_counts())

split
train         39089
test          10851
validation     4344
Name: count, dtype: int64


## 6. Xem kết quả

In [7]:
class_names = sorted(data["class_name"].unique())
class_to_id = {name: number for number, name in enumerate(class_names)}
data["class_id"] = data["class_name"].map(class_to_id)

# Chỉ lưu những cột mà model cần sử dụng.
model_data = data[[
    "relative_path",
    "class_name",
    "class_id",
    "split",
]].copy()

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
model_data.to_csv(PREPROCESSING_PATH, index=False)

print("Hoàn thành tiền xử lý!")
print(f"Tổng số ảnh: {len(model_data):,}")
print(f"Số lớp: {len(class_names)}")
print("Đã lưu:", PREPROCESSING_PATH)
print()
print(model_data["split"].value_counts())

model_data.head()

Hoàn thành tiền xử lý!
Tổng số ảnh: 54,284
Số lớp: 38
Đã lưu: C:\Users\MY PC\Documents\DL\project 1\plant-disease-classification\data\preprocessing_data.csv

split
train         39089
test          10851
validation     4344
Name: count, dtype: int64


,relative_path,class_name,class_id,split
0,train/Corn_(maize)___Northern_Leaf_Blight/47e4...,Corn_(maize)___Northern_Leaf_Blight,9,train
1,train/Tomato___Spider_mites Two-spotted_spider...,Tomato___Spider_mites Two-spotted_spider_mite,33,train
2,"train/Pepper,_bell___healthy/68c3951b-fb90-418...","Pepper,_bell___healthy",19,train
3,train/Tomato___Tomato_Yellow_Leaf_Curl_Virus/d...,Tomato___Tomato_Yellow_Leaf_Curl_Virus,35,train
4,"train/Pepper,_bell___Bacterial_spot/524b0b87-2...","Pepper,_bell___Bacterial_spot",18,train
